# Influence of the LLM-anomaly ratio and the alpha parameter

This notebook compares **SERAG** (`alpha = 0.5`) and **LoGNet** under the chronological and pooled protocols built from the same 30 ICEWS18 snapshots. The training-anomaly ratios are 1%, 2.5%, 5%, 10%, and 20%. Validation and test remain fixed at 10%. A second analysis studies SERAG for `alpha` values between 0 (relational branch only) and 1 (semantic branch only), at a fixed 10% ratio.

Each result is computed over seeds 41–45 and reported as **mean ± sample standard deviation**. SERAG and LoGNet checkpoints are selected on validation AUPRC.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:  # Also supports execution as a Python script.
    Markdown = str
    display = print
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

REPO_DIR = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "outputs").is_dir() and (path / "src").is_dir()
)
OUTPUTS_DIR = REPO_DIR / "outputs"
RATIOS = (0.01, 0.025, 0.05, 0.1, 0.2)
SEEDS = (41, 42, 43, 44, 45)
PROTOCOLS = ("chronological_30_llm", "pooled_30_llm")
MODELS = ("lognet", "serag")

def protocol_label(protocol: str) -> str:
    return "Chronological" if protocol == "chronological_30_llm" else "Pooled"

print("Repository located successfully.")

## Influence of the alpha parameter

This ablation uses a 10% anomaly ratio in all three splits. `alpha = 0` corresponds to the relational branch alone, and `alpha = 1` to the semantic branch alone. Intermediate values combine the two calibrated logits.

In [ ]:
ALPHAS = (0.0, 0.25, 0.5, 0.75, 0.85, 0.95, 1.0)
ALPHA_LABELS = {alpha: f"{alpha:g}" for alpha in ALPHAS}
alpha_frames = []
for protocol in PROTOCOLS:
    for alpha in ALPHAS:
        if protocol == "chronological_30_llm":
            path = (
                OUTPUTS_DIR / "cpu_llm_chronological_30" / "serag" / protocol
                / "ratio_0.1" / f"alpha_{alpha:g}" / "summary_by_seed.csv"
            )
        else:
            path = (
                OUTPUTS_DIR / "cpu_llm_alpha_sweep" / protocol / "ratio_0.1"
                / f"alpha_{alpha:g}" / "summary_by_seed.csv"
            )
        if not path.is_file():
            raise FileNotFoundError(path)
        frame = pd.read_csv(path)
        frame["protocol"] = protocol
        frame["alpha"] = alpha
        alpha_frames.append(frame)

alpha_results = pd.concat(alpha_frames, ignore_index=True)
alpha_completeness = (
    alpha_results.groupby(["protocol", "alpha"], as_index=False)
    .agg(n_seeds=("seed", "nunique"), n_ok=("status", lambda values: values.eq("ok").sum()))
)
assert len(alpha_results) == len(PROTOCOLS) * len(ALPHAS) * len(SEEDS)
assert alpha_completeness[["n_seeds", "n_ok"]].eq(len(SEEDS)).all().all()

def format_number(value: float) -> str:
    return f"{value:.3f}"

alpha_metrics = {"auc_test": "AUROC", "auprc_test": "AUPRC", "f1_test": "F1"}
alpha_aggregated = (
    alpha_results.groupby(["protocol", "alpha"])[list(alpha_metrics)]
    .agg(["mean", "std"])
)

def alpha_results_table(protocol: str) -> pd.DataFrame:
    rows = []
    for metric, label in alpha_metrics.items():
        row = {}
        for alpha in ALPHAS:
            values = alpha_aggregated.loc[(protocol, alpha), metric]
            row[ALPHA_LABELS[alpha]] = f"{format_number(values['mean'])} ± {format_number(values['std'])}"
        rows.append(row)
    return pd.DataFrame(rows, index=pd.Index(alpha_metrics.values(), name="Metric"))

for protocol in PROTOCOLS:
    label = protocol_label(protocol)
    display(alpha_results_table(protocol).style.set_caption(f"{label} — mean ± sample standard deviation over five seeds"))

alpha_selection = []
for protocol in PROTOCOLS:
    validation = alpha_results[alpha_results.protocol.eq(protocol)].groupby("alpha")["auprc_validation"].mean()
    selected_alpha = float(validation.idxmax())
    selected = alpha_results[alpha_results.protocol.eq(protocol) & alpha_results.alpha.eq(selected_alpha)]
    alpha_selection.append({
        "Protocol": protocol_label(protocol),
        "Alpha selected on validation": ALPHA_LABELS[selected_alpha],
        "Validation AUPRC": f"{format_number(selected.auprc_validation.mean())} ± {format_number(selected.auprc_validation.std())}",
        "Test AUPRC": f"{format_number(selected.auprc_test.mean())} ± {format_number(selected.auprc_test.std())}",
    })
display(pd.DataFrame(alpha_selection).style.hide(axis="index"))

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for row_index, protocol in enumerate(PROTOCOLS):
    for column_index, (metric, metric_label) in enumerate(alpha_metrics.items()):
        axis = axes[row_index, column_index]
        means = [alpha_aggregated.loc[(protocol, alpha), (metric, "mean")] for alpha in ALPHAS]
        stds = [alpha_aggregated.loc[(protocol, alpha), (metric, "std")] for alpha in ALPHAS]
        axis.errorbar(ALPHAS, means, yerr=stds, marker="o", capsize=4, linewidth=2, color="#1b9e77")
        axis.set_title(f"{protocol_label(protocol)} — {metric_label}")
        axis.set_xlabel("Alpha (0: relational; 1: semantic)")
        axis.set_ylabel(metric_label)
        axis.set_xticks(ALPHAS)
        axis.grid(alpha=0.25)
figure.suptitle("Influence of the semantic–relational combination", fontsize=15)
figure.tight_layout()
plt.show()

### Interpreting the ablation

- **Each isolated branch performs worse than the best fusion.** Under pooled evaluation, AUPRC increases from 0.257 (`alpha = 0`) and 0.275 (`alpha = 1`) to 0.321 (`alpha = 0.5`). Under chronological evaluation, it increases from 0.173 and 0.252 to 0.276 (`alpha = 0.75`).
- The semantic branch is more discriminative than the relational branch when each is used alone. Relative to `alpha = 1`, the validation-selected fusion improves AUPRC by 0.046 under pooled evaluation and 0.006 under chronological evaluation.
- Validation AUPRC selects `alpha = 0.5` in both protocols. This value also maximizes test AUPRC under pooled evaluation. Under chronological evaluation, `alpha = 0.75` reaches the highest test AUPRC, but it is a sensitivity result rather than a validation-selected configuration.
- The best F1 is obtained with `alpha = 0.5` under pooled evaluation and `alpha = 0.75` under chronological evaluation.

## Loading and validating results

In [ ]:
def result_path(model: str, protocol: str, ratio: float) -> Path:
    ratio_name = f"{ratio:g}"
    if protocol == "chronological_30_llm":
        base = OUTPUTS_DIR / "cpu_llm_chronological_30"
        if model == "lognet":
            return base / "lognet" / protocol / f"ratio_{ratio_name}" / "summary_by_seed.csv"
        if ratio == 0.1:
            return (
                base / "serag" / protocol / "ratio_0.1"
                / "alpha_0.5" / "summary_by_seed.csv"
            )
        return (
            base / "serag" / "alpha_0.5" / protocol
            / f"ratio_{ratio_name}" / "summary_by_seed.csv"
        )

    if model == "lognet":
        return (
            OUTPUTS_DIR / "cpu_llm_lognet_auprc" / protocol
            / f"ratio_{ratio_name}" / "summary_by_seed.csv"
        )
    if ratio == 0.1:
        return (
            OUTPUTS_DIR / "cpu_llm_alpha_sweep" / protocol / "ratio_0.1"
            / "alpha_0.5" / "summary_by_seed.csv"
        )

    base = OUTPUTS_DIR / "cpu_llm_ratio_sweep" / model
    if model == "serag":
        legacy_base = OUTPUTS_DIR / "cpu_llm_ratio_sweep" / "serad_kg"
        if not base.is_dir() and legacy_base.is_dir():
            base = legacy_base
        base = base / "alpha_0.5"
    return base / protocol / f"ratio_{ratio_name}" / "summary_by_seed.csv"


frames = []
for model in MODELS:
    for protocol in PROTOCOLS:
        for ratio in RATIOS:
            path = result_path(model, protocol, ratio)
            if not path.is_file():
                raise FileNotFoundError(path)
            frame = pd.read_csv(path)
            frame["model"] = model
            frame["protocol"] = protocol
            frame["ratio"] = ratio
            frames.append(frame)

results = pd.concat(frames, ignore_index=True)

completeness = (
    results.groupby(["protocol", "model", "ratio"], as_index=False)
    .agg(n_seeds=("seed", "nunique"), n_ok=("status", lambda values: values.eq("ok").sum()))
)
assert len(results) == len(PROTOCOLS) * len(MODELS) * len(RATIOS) * len(SEEDS)
assert completeness["n_seeds"].eq(len(SEEDS)).all()
assert completeness["n_ok"].eq(len(SEEDS)).all()
display(Markdown(f"**Loaded {len(results)} runs; every configuration contains five valid seeds.**"))
display(completeness)

In [ ]:
METRICS = {
    "auc_test": "AUROC",
    "auprc_test": "AUPRC",
    "precision_test": "Precision",
    "recall_test": "Recall",
    "f1_test": "F1",
}
MODEL_LABELS = {"lognet": "LoGNet", "serag": "SERAG"}
RATIO_LABELS = {0.01: "1%", 0.025: "2.5%", 0.05: "5%", 0.1: "10%", 0.2: "20%"}

aggregated = (
    results.groupby(["protocol", "model", "ratio"])[list(METRICS)]
    .agg(["mean", "std"])
    .sort_index()
)


def format_metric(value: float) -> str:
    return f"{value:.3f}"


def transposed_results_table(protocol: str) -> pd.DataFrame:
    rows = []
    index = []
    for model in MODELS:
        for metric, metric_label in METRICS.items():
            row = {}
            for ratio in RATIOS:
                values = aggregated.loc[(protocol, model, ratio), metric]
                row[RATIO_LABELS[ratio]] = (
                    f"{format_metric(values['mean'])} ± {format_metric(values['std'])}"
                )
            rows.append(row)
            index.append((MODEL_LABELS[model], metric_label))
    table = pd.DataFrame(
        rows,
        index=pd.MultiIndex.from_tuples(index, names=["Method", "Metric"]),
    )
    return table[[RATIO_LABELS[ratio] for ratio in RATIOS]]

## 1. Chronological protocol

The tables are transposed relative to the initial presentation: ratios appear in columns.

In [ ]:
chronological_table = transposed_results_table("chronological_30_llm")
display(chronological_table.style.set_caption("Chronological results — mean ± sample standard deviation over five seeds"))

## 2. Pooled protocol

In [ ]:
pooled_table = transposed_results_table("pooled_30_llm")
display(pooled_table.style.set_caption("Pooled results — mean ± sample standard deviation over five seeds"))

## Performance as the anomaly ratio changes

In [ ]:
plot_metrics = {"auc_test": "AUROC", "auprc_test": "AUPRC", "f1_test": "F1"}
colors = {"lognet": "#d95f02", "serag": "#1b9e77"}
markers = {"lognet": "s", "serag": "o"}

figure, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for row_index, protocol in enumerate(PROTOCOLS):
    for column_index, (metric, metric_label) in enumerate(plot_metrics.items()):
        axis = axes[row_index, column_index]
        for model in MODELS:
            means = [aggregated.loc[(protocol, model, ratio), (metric, "mean")] for ratio in RATIOS]
            stds = [aggregated.loc[(protocol, model, ratio), (metric, "std")] for ratio in RATIOS]
            axis.errorbar(
                range(len(RATIOS)), means, yerr=stds,
                marker=markers[model], capsize=4, linewidth=2,
                color=colors[model], label=MODEL_LABELS[model],
            )
        axis.set_title(f"{protocol_label(protocol)} — {metric_label}")
        axis.set_xticks(range(len(RATIOS)), [RATIO_LABELS[ratio] for ratio in RATIOS])
        axis.set_xlabel("Training-anomaly ratio")
        axis.set_ylabel(metric_label)
        axis.grid(alpha=0.25)
        if row_index == 0 and column_index == 0:
            axis.legend()
figure.suptitle("Influence of the training-anomaly ratio", fontsize=15)
figure.tight_layout()
plt.show()

## Selecting the ratio on validation data

In [ ]:
selection_rows = []
for protocol in PROTOCOLS:
    for model in MODELS:
        subset = results[(results["protocol"] == protocol) & (results["model"] == model)]
        validation_by_ratio = subset.groupby("ratio")["auprc_validation"].mean()
        selected_ratio = float(validation_by_ratio.idxmax())
        selected = subset[subset["ratio"] == selected_ratio]
        selection_rows.append(
            {
                "Protocol": protocol_label(protocol),
                "Method": MODEL_LABELS[model],
                "Selected ratio": RATIO_LABELS[selected_ratio],
                "Validation AUPRC": f"{format_metric(selected.auprc_validation.mean())} ± {format_metric(selected.auprc_validation.std(ddof=1))}",
                "Test AUROC": f"{format_metric(selected.auc_test.mean())} ± {format_metric(selected.auc_test.std(ddof=1))}",
                "Test AUPRC": f"{format_metric(selected.auprc_test.mean())} ± {format_metric(selected.auprc_test.std(ddof=1))}",
                "Test F1": f"{format_metric(selected.f1_test.mean())} ± {format_metric(selected.f1_test.std(ddof=1))}",
            }
        )

selection_table = pd.DataFrame(selection_rows)
display(selection_table.style.hide(axis="index").set_caption("Ratio selected exclusively on validation AUPRC"))

## Paired SERAG gains over LoGNet

In [ ]:
gain_rows = []
for protocol in PROTOCOLS:
    for ratio in RATIOS:
        serag = results[(results.protocol == protocol) & (results.model == "serag") & (results.ratio == ratio)].set_index("seed")
        lognet = results[(results.protocol == protocol) & (results.model == "lognet") & (results.ratio == ratio)].set_index("seed")
        row = {
            "Protocol": protocol_label(protocol),
            "Ratio": RATIO_LABELS[ratio],
        }
        for metric, label in (("auc_test", "Gain AUROC"), ("auprc_test", "Gain AUPRC"), ("f1_test", "Gain F1")):
            differences = serag[metric] - lognet[metric]
            interval = stats.t.interval(
                0.95, len(differences) - 1,
                loc=differences.mean(), scale=stats.sem(differences),
            )
            row[label] = (
                f"{format_metric(differences.mean())} "
                f"[{format_metric(interval[0])}, {format_metric(interval[1])}]"
            )
        gain_rows.append(row)

gain_table = pd.DataFrame(gain_rows)
display(gain_table.style.hide(axis="index").set_caption("Mean SERAG gain [95% CI]"))

## Conclusions

- **SERAG outperforms LoGNet at every ratio, under both protocols and across all five seeds.**
- SERAG improves steadily as the number of training anomalies increases; validation selects **20%** under both pooled and chronological evaluation.
- LoGNet reaches its best AUPRC at **2.5%** under pooled evaluation and **1%** under chronological evaluation.
- At 20%, SERAG reaches approximately **0.382 AUPRC and 0.405 F1** under pooled evaluation, and **0.289 AUPRC and 0.338 F1** under chronological evaluation.
- The alpha ablation shows that **fusion outperforms either isolated branch**. Validation selects `alpha = 0.5` in both protocols; `alpha = 0.75` obtains the highest chronological test AUPRC but is not selected on validation.
- The comparison evaluates the complete SERAG method against the adapted LoGNet reimplementation. SERAG uses class-balanced BCE, LoGNet uses standard BCE, and both select checkpoints on validation AUPRC.